# Stroke order filtering

Notebook helper to map one-character-per-line Chinese input to the existing stroke-order CSV, then derive the iframe embed HTML and image URL for each matched character.

In [1]:
from pathlib import Path
from typing import Iterable

import pandas as pd
from IPython.display import HTML, display

# CSV_PATH = Path(r"d:\repos\chinese-docs\docs\resources\List of Stroke Order Animation Embed URLs.csv")
CSV_PATH = Path("docs/resources/List of Stroke Order Animation Embed URLs.csv")
IMAGE_URL_TEMPLATE = "https://www.strokeorder.com/assets/bishun/guide/{id}.png"


def load_stroke_lookup(csv_path: Path = CSV_PATH) -> pd.DataFrame:
    df = pd.read_csv(csv_path, dtype=str)
    df.columns = ["no", "unicode_decimal", "character", "embed_html"]
    for column in df.columns:
        df[column] = df[column].astype(str).str.strip()
    df["image_url"] = df["unicode_decimal"].map(lambda value: IMAGE_URL_TEMPLATE.format(id=value))
    return df


def extract_characters(source: str | Path | Iterable[str]) -> list[str]:
    if isinstance(source, Path):
        text = source.read_text(encoding="utf-8")
    elif isinstance(source, str):
        if "\n" not in source and Path(source.strip()).exists():
            text = Path(source.strip()).read_text(encoding="utf-8")
        else:
            text = source
    else:
        text = "\n".join(str(item) for item in source)

    characters: list[str] = []
    seen: set[str] = set()
    for line in text.splitlines():
        character = line.strip()
        if len(character) != 1 or character in seen:
            continue
        seen.add(character)
        characters.append(character)
    return characters


def filter_stroke_entries(source: str | Path | Iterable[str], csv_path: Path = CSV_PATH) -> pd.DataFrame:
    characters = extract_characters(source)
    lookup = load_stroke_lookup(csv_path).set_index("character", drop=False)
    rows: list[dict[str, str]] = []

    for character in characters:
        if character not in lookup.index:
            continue
        record = lookup.loc[character]
        if isinstance(record, pd.DataFrame):
            record = record.iloc[0]
        rows.append(
            {
                "character": record["character"],
                "unicode_decimal": record["unicode_decimal"],
                "embed_html": record["embed_html"],
                "image_url": record["image_url"],
                "csv_no": record["no"],
            }
        )

    return pd.DataFrame(rows, columns=["character", "unicode_decimal", "embed_html", "image_url", "csv_no"])


def to_markdown_table(df: pd.DataFrame) -> str:
    if df.empty:
        return "| character | unicode_decimal | embed_html | image_url | csv_no |\n| --- | --- | --- | --- | --- |"

    header = "| " + " | ".join(df.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(df.columns)) + " |"
    lines = [header, separator]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(row[column]) for column in df.columns) + " |")
    return "\n".join(lines)


def render_results(source: str | Path | Iterable[str], csv_path: Path = CSV_PATH) -> pd.DataFrame:
    result = filter_stroke_entries(source, csv_path=csv_path)
    display(result)
    display(HTML(result.to_html(index=False, escape=False)))
    print(to_markdown_table(result))
    return result

In [2]:
sample_input = """
她
他
老
师
不
"""

result = render_results(sample_input)
print("First matched row:")
print(result.iloc[0].to_dict())
print("Zero image URL:")
# print(result.loc[result["character"] == "零", "image_url"].iloc[0])

output_csv = Path("stroke_filtering_output.csv")
result.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"Wrote {output_csv.resolve()}")

,character,unicode_decimal,embed_html,image_url,csv_no
0,她,22905,<iframe src='https://stroke-order.learningweb....,https://www.strokeorder.com/assets/bishun/guid...,373
1,他,20182,<iframe src='https://stroke-order.learningweb....,https://www.strokeorder.com/assets/bishun/guid...,175
2,老,32769,<iframe src='https://stroke-order.learningweb....,https://www.strokeorder.com/assets/bishun/guid...,448
3,师,24072,<image src='https://www.strokeorder.com/assets...,https://www.strokeorder.com/assets/bishun/guid...,24072
4,不,19981,<iframe src='https://stroke-order.learningweb....,https://www.strokeorder.com/assets/bishun/guid...,67


character,unicode_decimal,embed_html,image_url,csv_no
她,22905,,https://www.strokeorder.com/assets/bishun/guide/22905.png,373
他,20182,,https://www.strokeorder.com/assets/bishun/guide/20182.png,175
老,32769,,https://www.strokeorder.com/assets/bishun/guide/32769.png,448
师,24072,,https://www.strokeorder.com/assets/bishun/guide/24072.png,24072
不,19981,,https://www.strokeorder.com/assets/bishun/guide/19981.png,67


| character | unicode_decimal | embed_html | image_url | csv_no |
| --- | --- | --- | --- | --- |
| 她 | 22905 | <iframe src='https://stroke-order.learningweb.moe.edu.tw/dictFrame.jsp?ID=22905' frameborder=0 width=300 height=520 allow='fullscreen'></iframe> | https://www.strokeorder.com/assets/bishun/guide/22905.png | 373 |
| 他 | 20182 | <iframe src='https://stroke-order.learningweb.moe.edu.tw/dictFrame.jsp?ID=20182' frameborder=0 width=300 height=520 allow='fullscreen'></iframe> | https://www.strokeorder.com/assets/bishun/guide/20182.png | 175 |
| 老 | 32769 | <iframe src='https://stroke-order.learningweb.moe.edu.tw/dictFrame.jsp?ID=32769' frameborder=0 width=300 height=520 allow='fullscreen'></iframe> | https://www.strokeorder.com/assets/bishun/guide/32769.png | 448 |
| 师 | 24072 | <image src='https://www.strokeorder.com/assets/bishun/animation/24072.gif' width=300> | https://www.strokeorder.com/assets/bishun/guide/24072.png | 24072 |
| 不 | 19981 | <iframe src='https://stroke-order.lea